<a href="https://colab.research.google.com/github/rachmadnanda/MachineLearning2026/blob/main/jobsheet_pbl/JS01_PBL_244107020095_RACHMAD_FEBRIANANDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS01 — Pengenalan Lingkungan Kerja, Etika Data, dan Dataset PBL
## Proyek: RumahPoint — Prediksi Harga Properti Indonesia

Notebook ini adalah versi PBL dari JS01. Isi aslinya adalah instalasi library dan etika
penggunaan AI; di sini keduanya dikaitkan langsung dengan proyek **RumahPoint**, yang
memakai data hasil *scraping* portal properti.

## BAGIAN 1 — Lingkungan Kerja dan Library

Library yang dipakai sepanjang jobsheet PBL beserta fungsinya:

| Library | Fungsi dalam proyek |
|---|---|
| `pandas` | Membaca CSV, pembersihan, agregasi per lokasi |
| `numpy` | Operasi vektor/matriks dan transformasi log harga |
| `scikit-learn` | Preprocessing, clustering, klasifikasi, regresi, evaluasi |
| `matplotlib`, `seaborn` | Visualisasi distribusi dan hubungan antarfitur |
| `tensorflow` / `keras` | JST dan CNN untuk JS13–JS15 |
| `joblib` | Menyimpan model hasil pipeline (JS15) |
| `flask` | Deployment model sebagai API (JS15) |
| `annoy`, `faiss-cpu`, `hnswlib` | Pencarian tetangga terdekat (JS06) |
| `pyprep` | Standardisasi/otomatisasi praproses (dipakai pada jobsheet asli) |

Sel berikut memastikan semua library inti dapat diimpor dan mencatat versinya, supaya
hasil eksperimen bisa direproduksi.

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')

import sys, platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print(f'Python      : {platform.python_version()}')
print(f'pandas      : {pd.__version__}')
print(f'numpy       : {np.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'matplotlib  : {matplotlib.__version__}')
print(f'seaborn     : {sns.__version__}')

try:
    import tensorflow as tf
    print(f'tensorflow  : {tf.__version__}')
except ImportError:
    print('tensorflow  : belum terpasang (dibutuhkan mulai JS13)')

Python      : 3.12.14
pandas      : 3.1.1
numpy       : 2.5.4
scikit-learn: 1.9.1
matplotlib  : 3.11.2
seaborn     : 0.13.2


tensorflow  : 2.21.0


> Catatan: pada jobsheet asli, sel pertama berisi `!pip install pyprep`. Di lingkungan
> PBL, instalasi sebaiknya dilakukan sekali di awal (`pip install -r requirements.txt`),
> bukan diulang setiap kali notebook dijalankan, agar hasil eksperimen tidak berubah
> karena versi library yang berbeda.

## BAGIAN 2 — Etika dan Legalitas Data

Proyek ini memakai data properti yang dikumpulkan dari portal jual-beli rumah. Beberapa
hal yang harus diperhatikan sebelum data dipakai:

1. **Hak cipta dan ketentuan layanan.** Data iklan adalah milik portal/pemilik iklan.
   Pengambilan data harus mengikuti `robots.txt` dan ToS masing-masing situs; data tidak
   boleh dijual kembali atau dipublikasikan ulang mentah-mentah.
2. **Perlindungan data pribadi (UU PDP No. 27/2022).** Nama penjual, nomor telepon, dan
   tautan iklan termasuk data pribadi. Dataset yang dipakai di jobsheet ini hanya
   menyimpan atribut properti dan harga, tanpa identitas pemilik.
3. **Bias dan keterwakilan.** Sebagian besar data berasal dari Jabodetabek dan kota besar.
   Model yang dilatih pada data ini tidak otomatis akurat untuk daerah dengan sedikit
   data (lihat kembali analisis per kota di laporan UTS).
4. **Keterbatasan model.** Prediksi harga adalah estimasi statistik, bukan penilaian resmi.
   Untuk keputusan jual-beli, hasil model harus dibandingkan dengan harga pasar aktual.
5. **Transparansi.** Notebook ini mencatat sumber data dan aturan pembersihan agar hasil
   dapat diaudit dan direproduksi.

Sel berikut memeriksa kualitas dasar dataset: jumlah sumber, duplikat, dan nilai hilang —
langkah pertama sebelum data boleh dipakai untuk pelatihan.

In [2]:
DATA_PATH = 'dataset/properti_multi_kota.csv'
df_mentah = pd.read_csv(DATA_PATH)

print(f'Shape dataset mentah : {df_mentah.shape}')
print(f'Jumlah sumber unik   : {df_mentah["sumber"].nunique():,}')
print(f'Jumlah ejaan lokasi  : {df_mentah["lokasi"].nunique():,}')
print(f'Baris duplikat persis: {df_mentah.duplicated().sum():,}')
print()
print('Nilai hilang per kolom:')
print((df_mentah.isna().sum().to_frame('jumlah_hilang')
       .assign(persen=lambda x: (x['jumlah_hilang'] / len(df_mentah) * 100).round(2))
       .sort_values('jumlah_hilang', ascending=False).to_string()))

Shape dataset mentah : (64650, 8)
Jumlah sumber unik   : 513
Jumlah ejaan lokasi  : 595
Baris duplikat persis: 7,156

Nilai hilang per kolom:
                    jumlah_hilang  persen
usia_bangunan               59861   92.59
luas_bangunan                1604    2.48
jumlah_kamar_mandi            408    0.63
jumlah_kamar                  396    0.61
luas_m2                         0    0.00
lokasi                          0    0.00
sumber                          0    0.00
harga                           0    0.00


## BAGIAN 3 — Inspeksi Awal Dataset PBL

Dataset `properti_multi_kota.csv` adalah gabungan beberapa sumber (FLUS, 99.co,
rumah123, uas-properti) dengan kolom:

| Kolom | Arti |
|---|---|
| `sumber` | Asal data (portal/kota) |
| `lokasi` | Kota/kabupaten (ejaan belum seragam) |
| `luas_m2` | Luas tanah (m²) |
| `luas_bangunan` | Luas bangunan (m²), sebagian kosong |
| `jumlah_kamar` | Jumlah kamar tidur |
| `jumlah_kamar_mandi` | Jumlah kamar mandi |
| `usia_bangunan` | Usia bangunan (tahun); sebagian sumber menyimpan tahun bangunan |
| `harga` | Harga penawaran (Rupiah) |

Target proyek ada dua: **harga** (regresi) dan **kategori harga** Murah/Sedang/Mahal
(klasifikasi) yang dibentuk dari kuantil harga.

In [3]:
df_mentah.head()

,sumber,lokasi,luas_m2,luas_bangunan,jumlah_kamar,jumlah_kamar_mandi,usia_bangunan,harga
0,uas-properti/Jabodetabek,Tangerang,30.0,NaN,3.0,3.0,33.0,2.037543e+08
1,uas-properti/Jabodetabek,Bekasi,122.0,NaN,5.0,4.0,19.0,1.058032e+09
2,uas-properti/Jabodetabek,Bekasi,142.8,NaN,1.0,1.0,9.0,1.335793e+09
3,uas-properti/Jabodetabek,Bekasi,152.6,NaN,4.0,3.0,6.0,1.613771e+09
4,uas-properti/Jabodetabek,Depok,61.2,NaN,1.0,1.0,9.0,5.221562e+08


In [4]:
print('Tipe data tiap kolom:')
print(df_mentah.dtypes.to_string())
print()
print('Statistik deskriptif kolom numerik:')
df_mentah[['luas_m2', 'luas_bangunan', 'jumlah_kamar',
           'jumlah_kamar_mandi', 'usia_bangunan', 'harga']].describe().round(2)

Tipe data tiap kolom:
sumber                    str
lokasi                    str
luas_m2               float64
luas_bangunan         float64
jumlah_kamar          float64
jumlah_kamar_mandi    float64
usia_bangunan         float64
harga                 float64

Statistik deskriptif kolom numerik:


,luas_m2,luas_bangunan,jumlah_kamar,jumlah_kamar_mandi,usia_bangunan,harga
count,64650.00,6.304600e+04,6.425400e+04,6.424200e+04,4789.00,6.465000e+04
mean,1400.53,1.591196e+05,6.305890e+03,1.454074e+04,1310.07,1.019571e+14
std,275427.05,3.933645e+07,1.597736e+06,3.684999e+06,950.01,9.695374e+15
min,1.00,0.000000e+00,0.000000e+00,0.000000e+00,0.00,1.000000e+00
25%,72.00,4.000000e+01,2.000000e+00,1.000000e+00,27.00,3.770000e+08
50%,98.00,6.300000e+01,2.000000e+00,2.000000e+00,2010.00,7.391908e+08
75%,150.00,1.250000e+02,3.000000e+00,2.000000e+00,2021.00,1.645000e+09
max,70000000.00,9.876543e+09,4.050000e+08,9.340000e+08,2026.00,1.111111e+18


In [5]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].hist(df_mentah['harga'] / 1e9, bins=80, color='steelblue', edgecolor='white')
ax[0].set_xlabel('Harga (miliar Rp)')
ax[0].set_ylabel('Frekuensi')
ax[0].set_title('Distribusi harga (skala asli)')

ax[1].hist(np.log10(df_mentah['harga']), bins=80, color='darkorange', edgecolor='white')
ax[1].set_xlabel('log10(Harga)')
ax[1].set_ylabel('Frekuensi')
ax[1].set_title('Distribusi log10 harga')

plt.tight_layout()
plt.show()

print(f'Median harga : Rp {df_mentah["harga"].median():,.0f}')
print(f'Harga minimum: Rp {df_mentah["harga"].min():,.0f}')
print(f'Harga maksimum: Rp {df_mentah["harga"].max():,.0f}')

Median harga : Rp 739,190,786
Harga minimum: Rp 1
Harga maksimum: Rp 1,111,111,111,111,111,168


In [6]:
top_lokasi = df_mentah['lokasi'].value_counts().head(12)
top_sumber = df_mentah['sumber'].value_counts().head(8)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
top_lokasi[::-1].plot(kind='barh', ax=ax[0], color='teal')
ax[0].set_title('12 lokasi dengan iklan terbanyak')
ax[0].set_xlabel('Jumlah iklan')

top_sumber[::-1].plot(kind='barh', ax=ax[1], color='indianred')
ax[1].set_title('8 sumber data terbanyak')
ax[1].set_xlabel('Jumlah iklan')

plt.tight_layout()
plt.show()

**Analisis Bagian 3:**

- Dataset mentah berisi **64.650 baris dan 8 kolom** dari **513 sumber** (gabungan portal/kota) dengan **595 ejaan lokasi** dan **7.156 baris duplikat persis**. Setelah duplikat dan outlier dibuang pada JS02, tersisa **56.170 baris** yang dipakai konsisten di seluruh jobsheet PBL.
- Statistik mentah menunjukkan kesalahan entri yang serius: `luas_m2` maksimum **70.000.000 m²**, `luas_bangunan` mencapai **9,88 miliar**, harga minimum **Rp1**, dan harga maksimum **±Rp1,1 kuadriliun**. Nilai seperti ini bukan properti nyata, melainkan salah ketik/salah satuan, sehingga aturan pembersihan di JS02 (luas 10–2.000 m², harga Rp50 juta–Rp100 miliar) wajib diterapkan sebelum pemodelan.
- Harga sangat *right-skewed*: median **Rp739 juta**, sedangkan rata-rata jauh di atasnya karena ekor panjang. Karena itu regresi harga dilatih pada **log10(harga)**, bukan harga mentah, agar error tidak didominasi rumah puluhan miliar.
- `usia_bangunan` hilang pada **92,6% baris**. Sebagian sumber menyimpan **tahun** bangunan (mis. `2020`) sehingga perlu dikonversi menjadi usia. Kolom ini tidak dipakai langsung sebagai fitur; pada JS02 diturunkan menjadi `kondisi` (Baru/Bekas/Tidak diketahui) dengan ambang **3 tahun**.
- `lokasi` memiliki ratusan ejaan berbeda (mis. "Bekasi" vs "Kabupaten Bekasi"), sehingga perlu normalisasi/encoding sebelum dipakai model.
- `sumber` berguna untuk audit asal data, tetapi **tidak boleh** dipakai sebagai fitur prediksi: sumber berkorelasi dengan cara pengambilan data, bukan dengan nilai properti, dan berpotensi menjadi kebocoran informasi.
